[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Regularization and Sampling &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]
X_train, y_train = train[FEATURES], train["tmin"]
X_valid, y_valid = valid[FEATURES], valid["tmin"]
X_test, y_test = test[FEATURES], test["tmin"]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' helpers, rebuilt here from the same made-up stations so that each answer runs on its
own.

In [2]:
def lightgbm_validation(**settings):
    model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.1, random_state=0, verbose=-1, **settings)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
    return model.best_iteration_, round(float(model.best_score_["valid_0"]["l2"]), 3)


def xgboost_validation(**settings):
    model = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.1, early_stopping_rounds=100, random_state=0,
                             **settings)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    return model.best_iteration + 1, round(mean_squared_error(y_valid, model.predict(X_valid)), 3)

**1.** Train XGBoost with `reg_alpha=10`, the penalty on the size of each leaf's value, with `xgboost_validation`, and print its trees and validation loss beside the default's.

In [3]:
print("reg_alpha 10:", xgboost_validation(reg_alpha=10))
print("default:     ", xgboost_validation())

reg_alpha 10: (309, 2.537)
default:      (77, 2.526)


With `reg_alpha=10`, XGBoost stops after 309 trees with a validation loss of 2.537, against 77 trees and 2.526 with its defaults. The penalty makes every step smaller, so the model needs more of them to reach a similar loss.

**2.** Fit the XGBoost stump of the first worked example with `reg_lambda=100`, and check the value of its first leaf against the sum of that leaf's answers over its rows plus 100.

In [4]:
stump = xgb.XGBRegressor(n_estimators=1, max_depth=1, learning_rate=1.0, reg_lambda=100, base_score=0)
stump.fit(X_train, y_train)
nodes = stump.get_booster().trees_to_dataframe()
print(nodes[["Feature", "Split", "Gain", "Cover"]].round(3))

first = X_train[nodes.loc[0, "Feature"]] < nodes.loc[0, "Split"]
print("by hand:", round(y_train[first].sum() / (first.sum() + 100), 3))

       Feature  Split       Gain    Cover
0  day_of_year  100.0  217713.67  15000.0
1         Leaf   <NA>     -7.693   4039.0
2         Leaf   <NA>      0.814  10961.0
by hand: -7.693


With λ = 100 the stump splits at day 100 rather than 94, and the leaf for the days before it holds −7.693, which is their sum over their 4,039 rows plus 100.

**3.** Train LightGBM with `subsample=0.7` and `subsample_freq=1` from seeds 0, 1 and 2, and print the lowest and highest validation loss of the three.

In [5]:
losses = []
for seed in (0, 1, 2):
    model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.1, subsample=0.7, subsample_freq=1,
                              random_state=seed, verbose=-1)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
    losses.append(model.best_score_["valid_0"]["l2"])
print("lowest:", round(min(losses), 3), "| highest:", round(max(losses), 3))

lowest: 2.505 | highest: 2.558


From the three seeds the validation loss runs from 2.505 to 2.558, a spread of about 0.05 with a share of 0.7.

**4.** Train both libraries with `colsample_bytree=0.75`, and print their trees and validation losses.

In [6]:
print("LightGBM:", lightgbm_validation(colsample_bytree=0.75))
print("XGBoost: ", xgboost_validation(colsample_bytree=0.75))

LightGBM: (298, 2.572)
XGBoost:  (234, 2.471)


With three quarters of the columns, LightGBM's validation loss is 2.572 and XGBoost's 2.471, from one run each.

**5.** Train LightGBM with `num_leaves=8`, `min_child_samples=100` and `reg_lambda=10` together, and print its trees and validation loss.

In [7]:
print(lightgbm_validation(num_leaves=8, min_child_samples=100, reg_lambda=10))

(262, 2.404)


The three settings together give 262 trees and a validation loss of 2.404, the lowest in this notebook, and still a single run on one split, which the **Tuning** notebook would check with cross-validation.

---

&#8592; **Back to:** [Regularization and Sampling](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/07-regularization-and-sampling.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
